# v2401 因果价格历史读取模型 · fast 平台材料

上传本 Notebook 和同目录的 `v2401-model.json`，共两个 UTF-8 文本文件，保留文件名。
运行 `v2401-price-memory-s42`，固定使用验证选中的第 4 轮 EMA 权重；本地判定：**达到本地改善条件**。导出理由：本地验证主指标提高且覆盖未降低。

入口为 `main(datasources, start_date, end_date)`，`datasources` 提供 `bar1m`、`bar5m`、`bar30m` 的实际行情表名。返回 `date、instrument、factor`，覆盖请求股票池。日期字符串的结束日包含全天，时间戳的结束分钟也包含在内。导入不查询行情；下方受保护的调试单元格可用于短区间试运行。

本版完整保留父版75个输入通道，另接FP32对数价格中心和有效标志；在完整因果GRU历史上增加64维、4头价格历史读取路径，按每个查询时点最近有效闭合Bar定位历史。使用股票日共享编码和当前因果前缀；模型参数为2,641,743。输入精度遵循冻结配置，模型计算为FP32，输出为标准化目标单位的分数。

JSON 包含完整数值权重、归一化统计、配置、依赖及训练／推理源码。FP32 权重用九位有效数字保存，实际文本恢复已逐张量核对；统计量保留原精度。推理读取授权行情与股票池，训练标签直接使用已提供的 vwap_return，模型不依赖 exposure。

推理依赖：numpy>=1.26,<3, pandas>=2.2,<3, pyarrow>=14, torch>=2.6,<3，以及平台 `dai`。有 CUDA 时使用 GPU，否则使用 CPU。正式本地指标及选择依据保存在 JSON 的 metadata；本地验证范围与平台运行状态分别记录。

需要从头复现训练时，安装 JSON 所附训练依赖，提供授权原始 Parquet，并显式调用 `reproduce("reproduced-v2401", "/path/to/data", training_confirmed=True)`；该工作目录必须不存在。正常导入、评测入口和调试单元格均不启动训练。


In [ ]:
"""两文件提交的 Notebook 入口模板；导入时不读取数据、不创建附件。"""

_RUNTIME = None


def _bundle_directory():
    from pathlib import Path
    return Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()


def _read_bundle():
    import json
    raw = (_bundle_directory() / "v2401-model.json").read_bytes()
    return json.loads(raw.decode("utf-8-sig"))


def _load_modules(bundle, paths):
    import torch
    import builtins
    from pathlib import Path
    from types import ModuleType
    modules = {}

    def local_import(name, globals=None, locals=None, fromlist=(), level=0):
        if level == 0 and name in modules:
            return modules[name]
        return builtins.__import__(name, globals, locals, fromlist, level)

    for relative in paths:
        entry = bundle["sources"][relative]
        name = Path(relative).stem
        module = ModuleType(name)
        module.__file__ = str(_bundle_directory() / Path(relative).name)
        module.__dict__["__builtins__"] = dict(vars(builtins), __import__=local_import)
        modules[name] = module
        exec(compile(entry["text"], module.__file__, "exec"), module.__dict__)
    return modules


def _runtime():
    global _RUNTIME
    if _RUNTIME is None:
        bundle = _read_bundle()
        modules = _load_modules(bundle, bundle["runtime_sources"])
        _RUNTIME = bundle, modules
    return _RUNTIME


def main(datasources, start_date, end_date):
    bundle, modules = _runtime()
    return modules["v2401_predict"].predict(
        datasources, start_date, end_date, model_dir=_bundle_directory(), bundle=bundle,
    )


def reproduce(workspace, data_root, *, training_confirmed=False):
    """显式调用才启动预处理和训练；workspace 必须是不存在的新目录。"""
    if not training_confirmed:
        raise ValueError("v2401 训练复现尚待用户明确确认")
    bundle = _read_bundle()
    modules = _load_modules(bundle, ["solutions/v2401/v2401_reproduce.py"])
    return modules["v2401_reproduce"].reproduce(bundle, workspace, data_root,
                                           training_confirmed=True)


In [ ]:
if __name__ == "__main__":
    result = main(
        {f: "cpt_jyc_2026_stock_" + f for f in ["bar1m", "bar5m", "bar30m"]},
        "2024-12-30", "2024-12-31",
    )
    import numpy as np
    import pandas as pd
    assert list(result.columns) == ["date", "instrument", "factor"]
    assert not result.empty and result["factor"].dtype == np.dtype("float64")
    assert pd.api.types.is_datetime64_any_dtype(result["date"])
    assert result["instrument"].map(lambda value: isinstance(value, str)).all()
    assert not result[["date", "instrument"]].isna().any().any()
    assert not result.duplicated(["date", "instrument"]).any()
    assert np.isfinite(result["factor"]).all()
    print("输出行数：", len(result))
    print(result.dtypes)
    display(result.head())
